# Multilingual Layer Merging for Qwen3

CS 613 Natural Language Processing | Assignment 3

This notebook evaluates direct layer removal, Remove-and-Patch, Union-Select-Reconstruct (USR), and correlation-guided Fusion on Qwen3-0.6B in German, Spanish, and Japanese. Wiki-40B provides calibration, validation, and held-out language modeling data. MMMLU, INCLUDE, and MGSM measure downstream accuracy under a shared five-shot protocol. The unmodified Qwen3 checkpoint serves as the reference model.

## Execution

Run the cells in order with a Python 3.12 virtual environment selected as the notebook kernel. The setup cell installs the required packages in that environment. Model and dataset files are fetched from Hugging Face and cached locally. Full mode requires a CUDA-capable GPU.

- **Smoke mode** exercises the pipeline with reduced data and one merge stage per method.
- **Full mode** evaluates five merge stages per method and scores exactly 100,000 Wiki-40B test tokens per language.

Outputs are written to the corresponding mode directory under `assignment3_runs`. The `revisions.json` file records the model and dataset commits used by that run.

In [ ]:
# Install dependencies into the selected notebook kernel.
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

if sys.prefix == sys.base_prefix:
    raise RuntimeError("Select the project's .venv kernel before running this cell.")

pip = [sys.executable, "-m", "pip", "--disable-pip-version-check", "install"]
try:
    installed_torch = version("torch")
except PackageNotFoundError:
    installed_torch = None
if installed_torch != "2.8.0+cu128":
    subprocess.run(pip + ["--index-url", "https://download.pytorch.org/whl/cu128",
                          "torch==2.8.0+cu128"], check=True)
subprocess.run(pip + ["transformers==4.53.3", "datasets==3.6.0",
                      "pandas>=2,<3", "safetensors>=0.4.3", "matplotlib>=3.8,<4"], check=True)

In [ ]:
import gc
import hashlib
import json
import math
import os
import random
import re
import statistics
import time
import unicodedata
from contextlib import contextmanager
from pathlib import Path
from string import ascii_uppercase

# Use a repository-local cache when a local virtual environment is present.
if (Path.cwd() / ".venv").is_dir():
    os.environ.setdefault("HF_HOME", str(Path.cwd() / "work" / "huggingface"))
# Use HTTP downloads and allow the Windows cache to store regular files.
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
if os.name == "nt":
    os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

import pandas as pd
import torch
import torch.nn.functional as F
import transformers
from datasets import load_dataset
from huggingface_hub import HfApi, hf_hub_download
from IPython.display import display
from torch import nn
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

MODE = "smoke"  # Select the evaluation protocol.
assert MODE in {"full", "smoke"}
FULL = MODE == "full"
SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
         ) if DEVICE.type == "cuda" else torch.float32
if FULL and DEVICE.type != "cuda":
    raise RuntimeError("Use a CUDA runtime for the full experiment.")
MODEL = "Qwen/Qwen3-0.6B"
WIKI = "google/wiki40b"
MMMLU = "openai/MMMLU"
INCLUDE = "CohereLabs/include-base-44"
MGSM = "juletxara/mgsm"
LANGUAGES = {"de": ("DE-DE", "German"), "es": ("ES-LA", "Spanish"),
             "ja": ("JA-JP", "Japanese")}
FIT = 4096 if FULL else 128  # per language
VAL = 3072 if FULL else 128
TEST = 100_000 if FULL else 256
CAL_BLOCK = 128
CONTEXT, STRIDE = 512, 256
SHOTS = 5
PAIR_STARTS = (4, 8, 12, 16, 20, 24) if FULL else (16,)
MAX_STEPS = 5 if FULL else 1
RIDGES = (0.01, 0.1) if FULL else (0.1,)
RANKS = (32, 64) if FULL else (16,)
MATH_NEW_TOKENS = 256 if FULL else 32
OUT = Path("assignment3_runs") / MODE
OUT.mkdir(parents=True, exist_ok=True)

# Record immutable Hub revisions for reproducible runs.
revision_path = OUT / "revisions.json"
if revision_path.exists():
    revisions = json.loads(revision_path.read_text())
else:
    api = HfApi()
    revisions = {MODEL: api.model_info(MODEL).sha}
    revisions.update({repo: api.dataset_info(repo).sha for repo in (WIKI, MMMLU, INCLUDE, MGSM)})
    revision_path.write_text(json.dumps(revisions, indent=2))

In [ ]:
# Download the pinned Qwen3 weights and tokenizer into the Hugging Face cache.
weights_path = hf_hub_download(MODEL, filename="model.safetensors",
                               revision=revisions[MODEL])
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=revisions[MODEL])
print(f"Qwen3 weights ready ({Path(weights_path).stat().st_size / 1e9:.2f} GB)")
print(MODE, DEVICE, DTYPE, torch.__version__, transformers.__version__)
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name())
display(revisions)

def load_original():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL, revision=revisions[MODEL], torch_dtype=DTYPE,
        attn_implementation="sdpa", use_safetensors=True,
    ).to(DEVICE).eval().requires_grad_(False)
    assert len(model.model.layers) == 28
    assert set(model.config.layer_types) == {"full_attention"}
    return model

## Data and Perplexity

Wiki-40B splits are streamed and shuffled separately for each language. Corpus markup is removed and documents are separated by an EOS token. Equal token budgets give each language the same number of calibration tokens. An EOS prefix provides context for the first target. Overlapping context windows score each target once, including those in the final partial window.

Merge candidates are ranked by their mean change in validation negative log-likelihood across languages. Held-out perplexity is reported separately for each language. Differences in absolute perplexity across languages also reflect tokenization and corpus composition.

In [ ]:
def wiki_tokens(lang, split, budget):
    stream = load_dataset(WIKI, lang, split=split, streaming=True, revision=revisions[WIKI])
    stream = stream.shuffle(seed=SEED, buffer_size=256)
    tokens = []
    for row in stream:
        text = re.sub(r"_START_(ARTICLE|SECTION|PARAGRAPH)_|_NEWLINE_", "\n", row["text"]).strip()
        if text:
            tokens.extend(tokenizer.encode(text, add_special_tokens=False))
            tokens.append(tokenizer.eos_token_id)
        if len(tokens) >= budget:
            return torch.tensor(tokens[:budget], dtype=torch.long)
    raise ValueError(f"{lang}/{split} supplied only {len(tokens)} of {budget} tokens")

train_ids = {lang: wiki_tokens(lang, "train", FIT) for lang in LANGUAGES}
val_ids = {lang: wiki_tokens(lang, "validation", VAL) for lang in LANGUAGES}
calibration = [ids[offset:offset + CAL_BLOCK] for offset in range(0, FIT, CAL_BLOCK)
               for ids in train_ids.values()]

def windows(ids, eos, context=CONTEXT, stride=STRIDE):
    assert 0 < stride < context
    stream = torch.cat((ids.new_tensor([eos]), ids))
    for first in range(1, stream.numel(), stride):
        end = min(first + stride, stream.numel())
        start = max(0, end - context)
        yield stream[start:end - 1], stream[first:end]

@torch.inference_mode()
def mean_nll(model, ids):
    loss = 0.0
    count = 0
    for inputs, targets in windows(ids, tokenizer.eos_token_id):
        k = targets.numel()
        logits = model(inputs[None].to(DEVICE), use_cache=False, logits_to_keep=k).logits[0]
        loss += F.cross_entropy(logits.float(), targets.to(DEVICE), reduction="sum").item()
        count += k
    assert count == ids.numel() and count > 0
    return loss / count

def score_wiki(model, corpus):
    return {lang: mean_nll(model, ids) for lang, ids in corpus.items()}

def ppl(nll):
    return math.exp(nll) if math.isfinite(nll) and nll < 709 else math.inf

example = torch.arange(1031)
assert torch.equal(torch.cat([y for _, y in windows(example, -1)]), example)
assert all(len(x) <= CONTEXT - 1 and len(x) >= len(y) for x, y in windows(example, -1))

## Merge Operators

Direct removal discards the first block of the pair and retains the second. Remove-and-Patch replaces one block with a centered low-rank residual map fitted to that block's update. The retained decoder block maintains the attention cache.

USR and Fusion retain the second block's attention and RMSNorm layers. Both construct candidate SwiGLU features from the inputs observed after shortening the model. A fitted readout reconstructs the two-block teacher output after subtracting the retained block's post-attention residual. USR selects features by estimated contribution. Fusion processes neurons in descending activation variance, pairs them by the highest available Pearson correlation, and averages their gate and up weights. The fused activations are recomputed from those weights before fitting. Ridge regularization is scaled by the mean diagonal of the Gram matrix. All fits use the Wiki-40B training split.

In [ ]:
def hidden(output):
    return output[0] if isinstance(output, tuple) else output

def refresh_layers(model, layers):
    model.model.layers = nn.ModuleList(layers)
    model.config.num_hidden_layers = len(layers)
    model.config.layer_types = ["full_attention"] * len(layers)
    for index, layer in enumerate(layers):
        block = layer.block if isinstance(layer, PatchedBlock) else layer
        block.self_attn.layer_idx = index

@contextmanager
def temporary_pair(model, index, replacement):
    original = list(model.model.layers)
    refresh_layers(model, original[:index] + [replacement] + original[index + 2:])
    try:
        yield
    finally:
        refresh_layers(model, original)

class ResidualPatch(nn.Module):
    def __init__(self, center, shift, left, right):
        super().__init__()
        self.left = nn.Parameter(left, requires_grad=False)
        self.right = nn.Parameter(right, requires_grad=False)
        self.register_buffer("center", center)
        self.register_buffer("shift", shift)

    def forward(self, x):
        update = ((x.float() - self.center) @ self.left) @ self.right.T
        return (x.float() + self.shift + update).to(x.dtype)

class PatchedBlock(nn.Module):
    attention_type = "full_attention"

    def __init__(self, block, patch, before):
        super().__init__()
        self.block = block
        self.patch = patch
        self.before = before

    def forward(self, hidden_states, *args, **kwargs):
        if self.before:
            return self.block(self.patch(hidden_states), *args, **kwargs)
        result = self.block(hidden_states, *args, **kwargs)
        if isinstance(result, tuple):
            return (self.patch(result[0]), *result[1:])
        return self.patch(result)

@torch.inference_mode()
def capture(model, taps):
    records = {name: [] for name in taps}
    handles = []

    def record(name, tensor):
        records[name].append(tensor.reshape(-1, tensor.shape[-1]).float())

    def pre(name):
        return lambda module, args: record(name, args[0])

    def post(name):
        return lambda module, args, output: record(name, hidden(output))

    try:
        for name, (module, is_pre) in taps.items():
            handle = module.register_forward_pre_hook(pre(name)) if is_pre else module.register_forward_hook(post(name))
            handles.append(handle)
        for ids in calibration:
            model.model(ids[None].to(DEVICE), use_cache=False)
    finally:
        for handle in handles:
            handle.remove()
    return {name: torch.cat(values) for name, values in records.items()}

def ridge_solution(gram, cross, strength, prior=None):
    penalty = strength * gram.diag().mean().clamp_min(1e-8)
    regularized = gram.clone()
    regularized.diagonal().add_(penalty)
    rhs = cross if prior is None else cross + penalty * prior
    return torch.linalg.solve(regularized, rhs)

def patch_design(x, y, strength):
    update = y - x
    center = x.mean(0)
    shift = update.mean(0)
    xc = x - center
    linear = ridge_solution(xc.T @ xc, xc.T @ (update - shift), strength)
    fitted = xc @ linear
    _, axes = torch.linalg.eigh(fitted.T @ fitted)
    return center, shift, linear, axes

class RefitMLP(nn.Module):
    def __init__(self, gate, up, down, bias):
        super().__init__()
        self.gate = nn.Parameter(gate.to(DTYPE), requires_grad=False)
        self.up = nn.Parameter(up.to(DTYPE), requires_grad=False)
        self.down = nn.Parameter(down.T.contiguous().to(DTYPE), requires_grad=False)
        self.bias = nn.Parameter(bias.to(DTYPE), requires_grad=False)

    def forward(self, x):
        features = F.silu(F.linear(x, self.gate)) * F.linear(x, self.up)
        return F.linear(features, self.down, self.bias)

def feature_matrix(z, gate, up):
    # Use model precision for the feature calculation, including averaged weights.
    z, gate, up = z.to(DTYPE), gate.to(DTYPE), up.to(DTYPE)
    return (F.silu(F.linear(z, gate)) * F.linear(z, up)).float()

def correlated_pairs(features):
    centered = features - features.mean(0)
    scale = torch.linalg.vector_norm(centered, dim=0)
    normalized = centered / scale.clamp_min(1e-8)
    corr = (normalized.T @ normalized).cpu()
    available = torch.ones(corr.shape[0], dtype=torch.bool)
    pairs = []
    for i in scale.argsort(descending=True).tolist():
        if available[i]:
            available[i] = False
            j = int(corr[i].masked_fill(~available, -torch.inf).argmax())
            assert available[j]
            available[j] = False
            pairs.append((i, j))
    return torch.tensor(pairs, device=features.device).T

@torch.inference_mode()
def mlp_design(model, index, method, first, second, y2):
    with temporary_pair(model, index, second):
        norm = second.post_attention_layernorm
        data = capture(model, {"residual": (norm, True), "z": (norm, False)})
    z = data["z"]
    target = y2 - data["residual"]
    gate = torch.cat([b.mlp.gate_proj.weight.float() for b in (first, second)])
    up = torch.cat([b.mlp.up_proj.weight.float() for b in (first, second)])
    down = torch.cat([b.mlp.down_proj.weight.T.float() for b in (first, second)])
    width = gate.shape[0] // 2

    if method == "USR":
        energy = torch.zeros(len(gate), device=DEVICE)
        for batch in z.split(512):
            energy += feature_matrix(batch, gate, up).square().sum(0)
        selected = (energy * down.square().sum(1)).topk(width).indices.sort().values
        gate, up, prior = gate[selected], up[selected], down[selected]
    else:
        # Sample across the interleaved languages and across the training prefix.
        subset = z[::max(1, math.ceil(len(z) / 2048))]
        i, j = correlated_pairs(feature_matrix(subset, gate, up))
        gate, up, prior = (gate[i] + gate[j]) / 2, (up[i] + up[j]) / 2, down[i] + down[j]

    # Fit an intercept and ridge readout from sufficient statistics.
    total_f = torch.zeros(width, device=DEVICE)
    gram = torch.zeros((width, width), device=DEVICE)
    cross = torch.zeros((width, target.shape[1]), device=DEVICE)
    for batch, y in zip(z.split(512), target.split(512), strict=True):
        features = feature_matrix(batch, gate, up)
        total_f += features.sum(0)
        gram += features.T @ features
        cross += features.T @ y
    mean_f, mean_y = total_f / len(z), target.mean(0)
    gram -= torch.outer(total_f, mean_f)
    cross -= torch.outer(total_f, mean_y)
    return gate, up, prior, gram, cross, mean_f, mean_y

@torch.inference_mode()
def copy_with_mlp(block, mlp):
    # Share the attention and normalization modules while isolating the candidate MLP.
    import copy
    result = copy.copy(block)
    result._modules = block._modules.copy()
    result.mlp = mlp
    return result.eval()

def candidates(model, index, method, settings):
    first, second = model.model.layers[index:index + 2]
    if method == "Remove":
        yield {}, second
        return

    taps = {"y2": (second, False)}
    if method == "Patch":
        taps.update(x=(first.input_layernorm, True), y1=(first, False))
    state = capture(model, taps)
    if method == "Patch":
        directions = (True, False) if settings is None else (settings["patch_first"],)
        ridges = RIDGES if settings is None else (settings["ridge"],)
        ranks = RANKS if settings is None else (settings["rank"],)
        for before in directions:
            x, y = (state["x"], state["y1"]) if before else (state["y1"], state["y2"])
            for ridge in ridges:
                center, shift, linear, axes = patch_design(x, y, ridge)
                for rank in ranks:
                    right = axes[:, -rank:].contiguous()
                    patch = ResidualPatch(center, shift, linear @ right, right)
                    yield {"patch_first": before, "ridge": ridge, "rank": rank}, PatchedBlock(
                        second if before else first, patch, before).eval()
    else:
        gate, up, prior, gram, cross, mf, my = mlp_design(
            model, index, method, first, second, state["y2"])
        for ridge in (RIDGES if settings is None else (settings["ridge"],)):
            down = ridge_solution(gram, cross, ridge, prior)
            mlp = RefitMLP(gate, up, down, my - mf @ down)
            yield {"ridge": ridge}, copy_with_mlp(second, mlp)

## Validation Search and Compression Schedule

At each stage, the search evaluates unused, non-overlapping pairs on the current model and selects the candidate with the lowest finite mean change in validation negative log-likelihood. Each method retains the hyperparameters selected at the first stage. Full mode evaluates five stages per method. Smoke mode evaluates one.

First-stage grid results provide the ablation data. The notebook records candidate scores, hyperparameters, and the selected merge order. Downstream benchmarks compare the final stage of each method with the unmodified model.

In [ ]:
METHODS = ("Remove", "Patch", "USR", "Fusion")

def nparams(model):
    return sum(p.numel() for p in model.parameters())

@torch.inference_mode()
def search(model, method, original_val):
    positions = [(i,) for i in range(len(model.model.layers))]
    remaining = list(PAIR_STARTS)
    trials, plan = [], []
    settings = None
    for step in range(1, MAX_STEPS + 1):
        best = None
        for start in remaining:
            index = positions.index((start,))
            assert positions[index + 1] == (start + 1,)
            for config, replacement in candidates(model, index, method, settings):
                with temporary_pair(model, index, replacement):
                    val = score_wiki(model, val_ids)
                deltas = {lang: val[lang] - original_val[lang] for lang in LANGUAGES}
                mean_delta = statistics.mean(deltas.values())
                finite = math.isfinite(mean_delta)
                row = dict(method=method, step=step, pair_start=start, **config,
                           mean_delta_nll=mean_delta, finite=finite, chosen=False,
                           **{f"ppl_{l}": ppl(val[l]) for l in LANGUAGES})
                trials.append(row)
                if finite and (best is None or mean_delta < best[0]):
                    best = mean_delta, row, index, config, replacement
        if best is None:
            raise FloatingPointError(f"{method} stage {step}: no finite validation score")
        _, row, index, config, _ = best
        row["chosen"] = True
        start = row["pair_start"]
        plan.append(dict(pair_start=start, **config))
        settings = config
        # Reuse the chosen candidate directly, in the order in which it was validated.
        layers = list(model.model.layers)
        refresh_layers(model, layers[:index] + [best[-1]] + layers[index + 2:])
        positions[index:index + 2] = [(start, start + 1)]
        remaining.remove(start)
        print(method, step, plan[-1], "parameters:", nparams(model))
    return trials, plan

@torch.inference_mode()
def replay(model, method, plan):
    # Map original pair indices to the current decoder positions.
    seen = []
    for item in plan:
        start = item["pair_start"]
        index = start - sum(previous < start for previous in seen)
        settings = {key: value for key, value in item.items() if key != "pair_start"}
        config, replacement = next(candidates(model, index, method, settings))
        assert config == settings
        layers = list(model.model.layers)
        refresh_layers(model, layers[:index] + [replacement] + layers[index + 2:])
        seen.append(start)
        yield model

def cleanup():
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

original = load_original()
original_val = score_wiki(original, val_ids)
assert all(math.isfinite(x) for x in original_val.values())
original_params = nparams(original)
print("Original validation PPL", {lang: ppl(nll) for lang, nll in original_val.items()})
del original
cleanup()

all_trials, plans = [], {}
for method in METHODS:
    model = load_original()
    trials, plans[method] = search(model, method, original_val)
    all_trials.extend(trials)
    del model
    cleanup()
    pd.DataFrame(all_trials).to_csv(OUT / "validation_trials.csv", index=False)
    (OUT / "merge_plans.json").write_text(json.dumps(plans, indent=2))
display(pd.DataFrame(all_trials).query("step == 1"))

## Few-Shot Benchmarks

The reference checkpoint and all merged variants receive identical tokenized prompts with five demonstrations per item. Qwen3 thinking mode is disabled.

- **MMMLU.** A seeded shuffle reserves five demonstrations from the released test file for each language. The reserved rows and exact question duplicates are excluded from evaluation. Full mode scores 500 items per language using seeded round-robin sampling across subjects.
- **INCLUDE-base-44.** Five demonstrations come from the official validation split. Evaluation uses the test split after excluding exact matches to demonstration questions.
- **MGSM.** Five demonstrations come from the official training split. Evaluation uses the official test split. Prompts request a final line of the form `#### 42`. Scoring accepts that format or a bare integer after Unicode normalization. Unparseable outputs are counted separately and scored as incorrect.

Multiple-choice predictions use the highest next-token logit among the four answer labels. MGSM uses greedy decoding and numeric exact match. Smoke mode scores two items per task and language.

## Benchmark Protocol and Interpretation

All models receive the same five demonstrations and tokenized prompts for each item. MMMLU reserves demonstrations from the released test file and scores a seeded sample of the remaining questions, balanced across subjects. INCLUDE uses validation demonstrations and the test split. MGSM uses training demonstrations, direct-answer prompting, greedy decoding, and numeric exact match. This MGSM setting uses no chain-of-thought demonstrations. These choices differ from some published benchmark protocols and affect comparisons with leaderboard scores.

The protocol and sample counts are recorded alongside the predictions. Accuracy changes are measured in percentage points relative to the original checkpoint.


In [ ]:
LABELS = ascii_uppercase[:4]
INSTRUCTIONS = {
    "de": ("Wähle die richtige Antwort. Antworte nur mit A, B, C oder D.",
           "Löse die Aufgabe. Schreibe das Ergebnis als ganze Zahl in die letzte Zeile, im Format #### 42."),
    "es": ("Elige la respuesta correcta. Responde solo con A, B, C o D.",
           "Resuelve el problema. Escribe el resultado entero en la última línea con el formato #### 42."),
    "ja": ("正しい選択肢を選び、A、B、C、D のいずれかだけを出力してください。",
           "問題を解き、最後の行に整数の答えを #### 42 の形式で書いてください。"),
}

def clean_question(text):
    return " ".join(unicodedata.normalize("NFKC", str(text)).casefold().split())

def mmlu_row(row):
    answer = str(row["Answer"]).strip().upper()
    assert answer in tuple(LABELS)
    return str(row["Question"]), [str(row[c]) for c in LABELS], answer

def include_row(row):
    answer = int(row["answer"])
    assert 0 <= answer < 4
    return row["question"], [row[f"option_{c.lower()}"] for c in LABELS], LABELS[answer]

def math_row(row):
    return row["question"], None, int(row["answer_number"])

def make_prompt(lang, task, examples, row):
    def content(item):
        question, choices, _ = item
        options = "" if choices is None else "\n" + "\n".join(
            f"{label}. {choice}" for label, choice in zip(LABELS, choices, strict=True))
        return question + options

    messages = [{"role": "system", "content": INSTRUCTIONS[lang][task == "MGSM"]}]
    for shot in examples:
        answer = f"#### {shot[-1]}" if task == "MGSM" else str(shot[-1])
        messages.extend([{"role": "user", "content": content(shot)},
                         {"role": "assistant", "content": answer}])
    messages.append({"role": "user", "content": content(row)})
    ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                        enable_thinking=False)
    assert isinstance(ids, list) and ids and isinstance(ids[0], int)
    assert len(ids) + MATH_NEW_TOKENS <= 40960, "Prompt exceeds context budget"
    return ids

def build_benchmarks():
    items = []
    for lang, (locale, include_name) in LANGUAGES.items():
        csv_path = hf_hub_download(MMMLU, f"test/mmlu_{locale}.csv",
                                   repo_type="dataset", revision=revisions[MMMLU])
        mmlu = pd.read_csv(csv_path, keep_default_na=False)
        ids = list(range(len(mmlu)))
        random.Random(SEED).shuffle(ids)
        mmlu_examples = [mmlu_row(mmlu.iloc[i]) for i in ids[:SHOTS]]
        example_questions = {clean_question(item[0]) for item in mmlu_examples}
        by_subject = {}
        for i in ids[SHOTS:]:
            row = mmlu.iloc[i]
            if clean_question(row["Question"]) not in example_questions:
                by_subject.setdefault(str(row["Subject"]), []).append(i)
        subject_names = sorted(by_subject)
        mmlu_eval_ids = []
        while len(mmlu_eval_ids) < (500 if FULL else 2):
            before = len(mmlu_eval_ids)
            for subject in subject_names:
                if by_subject[subject]:
                    mmlu_eval_ids.append(by_subject[subject].pop())
                if len(mmlu_eval_ids) == (500 if FULL else 2):
                    break
            if len(mmlu_eval_ids) == before:
                break

        include = load_dataset(INCLUDE, include_name, revision=revisions[INCLUDE])
        mgsm = load_dataset(MGSM, lang, revision=revisions[MGSM])
        groups = [
            ("MMMLU", mmlu_examples, ((i, mmlu_row(mmlu.iloc[i])) for i in mmlu_eval_ids),
             500 if FULL else 2),
            ("INCLUDE", [include_row(r) for r in include["validation"].select(range(SHOTS))],
             ((i, include_row(r)) for i, r in enumerate(include["test"])), None if FULL else 2),
            ("MGSM", [math_row(r) for r in mgsm["train"].select(range(SHOTS))],
             ((i, math_row(r)) for i, r in enumerate(mgsm["test"])), None if FULL else 2),
        ]
        for task, examples, rows, limit in groups:
            assert len(examples) == SHOTS
            demo_questions = {clean_question(r[0]) for r in examples}
            added = 0
            for row_id, row in rows:
                if clean_question(row[0]) in demo_questions:
                    continue
                token_ids = make_prompt(lang, task, examples, row)
                digest = hashlib.sha256(json.dumps(token_ids).encode()).hexdigest()
                items.append(dict(task=task, language=lang, row_id=row_id,
                                  ids=token_ids, prompt_sha256=digest, gold=row[-1]))
                added += 1
                if limit is not None and added == limit:
                    break
            assert added > 0 and (limit is None or added == limit), (task, lang, added)
    return items

def integer_answer(text):
    normalized = unicodedata.normalize("NFKC", text).strip().replace("−", "-")
    last_line = normalized.rsplit("\n", 1)[-1].strip()
    if last_line.startswith("####"):
        normalized = last_line[4:].strip()
    if re.fullmatch(r"[+-]?(?:\d{1,3}(?:,\d{3})+|\d+)", normalized):
        return int(normalized.replace(",", ""))
    return None

def padded(items):
    sequences = [item["ids"] for item in items]
    width = max(map(len, sequences))
    ids = torch.full((len(sequences), width), tokenizer.pad_token_id, dtype=torch.long)
    mask = torch.zeros_like(ids)
    for i, sequence in enumerate(sequences):
        ids[i, -len(sequence):] = torch.tensor(sequence)
        mask[i, -len(sequence):] = 1
    return {"input_ids": ids.to(DEVICE), "attention_mask": mask.to(DEVICE)}

@torch.inference_mode()
def evaluate_benchmarks(model, items, method, steps):
    label_ids = [tokenizer.encode(c, add_special_tokens=False) for c in LABELS]
    assert all(len(ids) == 1 for ids in label_ids)
    assert len({ids[0] for ids in label_ids}) == len(LABELS)
    choice_ids = [ids[0] for ids in label_ids]
    generation = GenerationConfig(
        do_sample=False, max_new_tokens=MATH_NEW_TOKENS, use_cache=True,
        eos_token_id=model.generation_config.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    predictions = []
    for task in ("MMMLU", "INCLUDE", "MGSM"):
        group = [item for item in items if item["task"] == task]
        batch_size = 8 if DEVICE.type == "cuda" else 2
        for start in range(0, len(group), batch_size):
            batch = group[start:start + batch_size]
            inputs = padded(batch)
            if task == "MGSM":
                sequences = model.generate(**inputs, generation_config=generation, use_model_defaults=False)
                raw = tokenizer.batch_decode(sequences[:, inputs["input_ids"].shape[1]:],
                                             skip_special_tokens=True)
                guesses = [integer_answer(s) for s in raw]
            else:
                logits = model(**inputs, use_cache=False, logits_to_keep=1).logits
                guesses = [LABELS[i] for i in logits[:, -1, choice_ids].argmax(-1).tolist()]
                raw = guesses
            for item, guess, text in zip(batch, guesses, raw, strict=True):
                predictions.append(dict(
                    method=method, steps=steps, task=task, language=item["language"],
                    row_id=item["row_id"], prompt_sha256=item["prompt_sha256"],
                    gold=item["gold"], prediction=guess, raw_output=text,
                    correct=guess == item["gold"],
                ))
    return predictions

## Held-Out Evaluation and Latency

Wiki-40B perplexity and efficiency metrics are recorded at each compression stage. The unmodified checkpoint and each method's final model are evaluated on the same downstream items. Batch-one latency is measured on an identical prompt for both prefill and fixed-length generation of 32 tokens. The fixed output length makes the timing comparison independent of when a model emits an EOS token.

In [ ]:
def sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

@torch.inference_mode()
def latency(model, ids, repeats=None, warmups=2):
    x = ids[None, :256].to(DEVICE)
    mask = torch.ones_like(x)
    count = 32
    # Suppress EOS until the fixed timing workload is complete.
    config = GenerationConfig(do_sample=False, use_cache=True, max_new_tokens=count,
                              min_new_tokens=count, pad_token_id=tokenizer.pad_token_id,
                              eos_token_id=model.generation_config.eos_token_id)
    def prefill():
        return model(x, attention_mask=mask, use_cache=True, logits_to_keep=1)

    def generate():
        return model.generate(x, attention_mask=mask, generation_config=config, use_model_defaults=False)

    out = prefill()
    assert out.past_key_values.get_seq_length() == x.shape[1]
    assert generate().shape[1] == x.shape[1] + count
    del out
    repeats = (7 if FULL else 3) if repeats is None else repeats
    readings = {}
    for name, run in [("prefill_ms", prefill), ("generate_32_ms", generate)]:
        for _ in range(warmups):
            run()
        times = []
        for _ in range(repeats):
            sync()
            t0 = time.perf_counter()
            run()
            sync()
            times.append((time.perf_counter() - t0) * 1000)
        readings[name] = statistics.median(times)
    return readings

test_ids = {lang: wiki_tokens(lang, "test", TEST) for lang in LANGUAGES}
def latency_excerpts(corpus, count, width):
    excerpts = []
    for language, ids in corpus.items():
        assert len(ids) >= count * width
        for excerpt in range(count):
            offset = excerpt * (len(ids) - width) // (count - 1)
            excerpts.append((language, excerpt + 1, offset, ids[offset:offset + width]))
    return excerpts


latency_inputs = latency_excerpts(test_ids, count=4 if FULL else 2, width=128)
latency_rows = []


def record_multilingual_latency(model, method, stage):
    for language, excerpt, offset, ids in latency_inputs:
        measurements = latency(model, ids, repeats=3 if FULL else 1, warmups=0)
        latency_rows.extend(
            dict(method=method, steps=stage, language=language, excerpt=excerpt,
                 token_offset=offset, prompt_tokens=len(ids), metric=metric, latency_ms=ms)
            for metric, ms in measurements.items()
        )
    pd.DataFrame(latency_rows).to_csv(OUT / "multilingual_latency_samples.csv", index=False)


benchmark_items = build_benchmarks()
(OUT / "prompt_manifest.json").write_text(json.dumps(benchmark_items, ensure_ascii=False),
                                          encoding="utf-8")
benchmark_counts = pd.DataFrame(benchmark_items).groupby(["task", "language"]).size()
print(benchmark_counts.rename("examples"))
benchmark_counts.rename("examples").reset_index().to_csv(OUT / "benchmark_sample_counts.csv", index=False)
benchmark_protocol = {
    "seed": SEED, "shots": SHOTS, "mode": MODE, "thinking_enabled": False,
    "mmmlu": {
        "source": MMMLU,
        "demonstrations": "Five seeded reserved rows from each released language test file",
        "evaluation": "Seeded round-robin sample by subject after excluding exact demonstration-question matches",
        "maximum_items_per_language": 500 if FULL else 2,
        "scoring": "Largest next-token logit among A, B, C and D",
        "standard_leaderboard_protocol": False,
    },
    "include": {
        "source": INCLUDE,
        "demonstrations": "First five validation rows for each language",
        "evaluation": "Test rows excluding exact demonstration-question matches",
        "maximum_items_per_language": None if FULL else 2,
        "scoring": "Largest next-token logit among A, B, C and D",
    },
    "mgsm": {
        "source": MGSM,
        "demonstrations": "First five training rows for each language",
        "evaluation": "Test rows excluding exact demonstration-question matches",
        "maximum_items_per_language": None if FULL else 2,
        "scoring": "Numeric exact match with final #### marker or bare integer",
        "prompting": "Five-shot direct answer without chain-of-thought demonstrations",
        "decoding": "Greedy", "max_new_tokens": MATH_NEW_TOKENS,
    },
}
(OUT / "benchmark_protocol.json").write_text(
    json.dumps(benchmark_protocol, indent=2, ensure_ascii=False), encoding="utf-8")
wiki_rows, efficiency_rows, predictions = [], [], []

def record_wiki(model, method, stage, path):
    scores = score_wiki(model, test_ids)
    params = nparams(model)
    wiki_rows.extend(dict(method=method, steps=stage, language=lang,
                          target_tokens=len(test_ids[lang]), nll=nll, ppl=ppl(nll))
                     for lang, nll in scores.items())
    efficiency_rows.append(dict(
        method=method, steps=stage, pair_order=[item["pair_start"] for item in path],
        attention_blocks=len(model.model.layers), parameters=params,
        buffer_values=sum(b.numel() for b in model.buffers()),
        reduction_pct=100 * (1 - params / original_params),
        **latency(model, test_ids["de"]),
    ))
    pd.DataFrame(wiki_rows).to_csv(OUT / "wiki_test.csv", index=False)
    pd.DataFrame(efficiency_rows).to_csv(OUT / "efficiency.csv", index=False)
    print(method, stage, {lang: round(ppl(nll), 2) for lang, nll in scores.items()})

def record_benchmarks(model, method, stage):
    predictions.extend(evaluate_benchmarks(model, benchmark_items, method, stage))
    pd.DataFrame(predictions).to_csv(OUT / "predictions.csv", index=False)

model = load_original()
record_wiki(model, "Original", 0, [])
record_benchmarks(model, "Original", 0)
record_multilingual_latency(model, "Original", 0)
del model
cleanup()
for method, plan in plans.items():
    assert len(plan) == MAX_STEPS, f"Rerun validation search for {method}"
    model = load_original()
    for stage, _ in enumerate(replay(model, method, plan), 1):
        record_wiki(model, method, stage, plan[:stage])
    del _
    record_benchmarks(model, method, len(plan))
    record_multilingual_latency(model, method, len(plan))
    del model
    cleanup()

wiki = pd.DataFrame(wiki_rows)
original_wiki = wiki.query("method == 'Original'").set_index("language")["nll"]
wiki["ppl_change_pct"] = 100 * (wiki["nll"].sub(wiki["language"].map(original_wiki)).map(ppl) - 1)
wiki.to_csv(OUT / "wiki_test.csv", index=False)
display(wiki.pivot(index=["method", "steps"], columns="language", values="ppl").round(2))

results = pd.DataFrame(predictions)
key = ["task", "language", "row_id", "prompt_sha256"]
reference_keys = set(results.query("method == 'Original'")[key].itertuples(index=False, name=None))
for method, group in results.groupby("method"):
    actual_keys = set(group[key].itertuples(index=False, name=None))
    assert actual_keys == reference_keys and len(group) == len(reference_keys), method
results["unparsed"] = results["prediction"].isna()
accuracy = results.groupby(["method", "steps", "task", "language"]).agg(
    examples=("correct", "size"), accuracy=("correct", "mean"),
    unparsed=("unparsed", "sum")).reset_index()
accuracy["accuracy_pct"] = 100 * accuracy.pop("accuracy")
baseline = accuracy.query("method == 'Original'")[["task", "language", "accuracy_pct"]]
accuracy = accuracy.merge(baseline.rename(columns={"accuracy_pct": "original_pct"}),
                          on=["task", "language"], validate="many_to_one")
accuracy["delta_pp"] = accuracy["accuracy_pct"] - accuracy["original_pct"]
accuracy.to_csv(OUT / "benchmark_accuracy.csv", index=False)
display(accuracy.round(2))
display(pd.DataFrame(efficiency_rows).round(2))

## Calibration Language Ablation

For each fitted method, the first layer pair and hyperparameters selected by the balanced multilingual search are held fixed. Two versions are fitted from the original checkpoint with equal total Wiki-40B training-token budgets.

- German-only calibration uses 12,288 German tokens in full mode.
- Balanced multilingual calibration uses 4,096 tokens each in German, Spanish, and Japanese in full mode.

Smoke mode uses 384 tokens per condition. The German-only sample shares its initial German tokens with the balanced condition. Both versions are evaluated on the same held-out Wiki-40B tokens and frozen benchmark prompts as the original model. Since the pair and hyperparameters were selected using balanced validation, the ablation compares calibration data at that fixed configuration. Direct removal has no fitted calibration parameters and is excluded.

In [ ]:
balanced_calibration = calibration
german_ids = wiki_tokens("de", "train", FIT * len(LANGUAGES))
assert torch.equal(german_ids[:FIT], train_ids["de"])
german_calibration = [german_ids[offset:offset + CAL_BLOCK]
                      for offset in range(0, len(german_ids), CAL_BLOCK)]
calibration_sets = {"German only": german_calibration,
                    "Balanced multilingual": balanced_calibration}
calibration_budget = FIT * len(LANGUAGES)
assert all(sum(chunk.numel() for chunk in chunks) == calibration_budget
           for chunks in calibration_sets.values())

ablation_wiki_rows = []
ablation_predictions = []
try:
    for method in ("Patch", "USR", "Fusion"):
        first = plans[method][0]
        pair_start = first["pair_start"]
        settings = {key: value for key, value in first.items() if key != "pair_start"}
        for condition, calibration in calibration_sets.items():
            model = load_original()
            config, replacement = next(candidates(model, pair_start, method, settings))
            assert config == settings
            layers = list(model.model.layers)
            refresh_layers(model, layers[:pair_start] + [replacement] + layers[pair_start + 2:])
            scores = score_wiki(model, test_ids)
            for language, nll in scores.items():
                ablation_wiki_rows.append(dict(
                    method=method, calibration=condition, pair_start=pair_start,
                    calibration_tokens=calibration_budget, language=language,
                    test_tokens=len(test_ids[language]), nll=nll, ppl=ppl(nll),
                    original_ppl=ppl(original_wiki[language]),
                    ppl_change_pct=100 * (ppl(nll - original_wiki[language]) - 1),
                    parameters=nparams(model), settings=json.dumps(settings, sort_keys=True),
                ))
            rows = evaluate_benchmarks(model, benchmark_items, method, 1)
            for row in rows:
                row["calibration"] = condition
            actual = {(row["task"], row["language"], row["row_id"], row["prompt_sha256"])
                      for row in rows}
            assert actual == reference_keys and len(rows) == len(reference_keys)
            ablation_predictions.extend(rows)
            pd.DataFrame(ablation_wiki_rows).to_csv(OUT / "calibration_wiki.csv", index=False)
            pd.DataFrame(ablation_predictions).to_csv(OUT / "calibration_predictions.csv", index=False)
            print(method, condition, "pair", pair_start,
                  {lang: round(ppl(value), 2) for lang, value in scores.items()})
            del model, layers, replacement
            cleanup()
finally:
    calibration = balanced_calibration

calibration_wiki = pd.DataFrame(ablation_wiki_rows)
calibration_predictions = pd.DataFrame(ablation_predictions)
calibration_predictions["unparsed"] = calibration_predictions["prediction"].isna()
calibration_accuracy = calibration_predictions.groupby(
    ["method", "calibration", "task", "language"]
).agg(examples=("correct", "size"), accuracy_pct=("correct", lambda values: 100 * values.mean()),
      unparsed=("unparsed", "sum")).reset_index()
calibration_accuracy = calibration_accuracy.merge(
    baseline.rename(columns={"accuracy_pct": "original_pct"}),
    on=["task", "language"], validate="many_to_one")
calibration_accuracy["delta_pp"] = (
    calibration_accuracy["accuracy_pct"] - calibration_accuracy["original_pct"])
calibration_accuracy.to_csv(OUT / "calibration_accuracy.csv", index=False)
display(calibration_wiki[
    ["method", "calibration", "pair_start", "language", "test_tokens", "ppl", "ppl_change_pct"]
].round(2))
display(calibration_accuracy.round(2))

## Multilingual Inference Benchmark

The original checkpoint and each final merged model receive the same 128-token excerpts from the held-out Wiki-40B stream in German, Spanish, and Japanese. Excerpts are non-overlapping and spaced evenly across the sampled stream. Full mode uses four excerpts per language and three timed trials per excerpt. Smoke mode uses two excerpts per language and one timed trial per excerpt.

Each workload runs at batch size one with the attention cache enabled. Latency is recorded separately for prefill and for generation of exactly 32 tokens including prefill time. An untimed pass verifies the workload before timing. The table reports the median and interquartile range across excerpt-level medians. Speedup is the median of the original-to-merged latency ratios on paired excerpts. Values greater than 1 indicate lower latency than the original checkpoint.

In [ ]:
def summarize_multilingual_latency(samples):
    keys = ["language", "excerpt", "token_offset", "prompt_tokens", "metric"]
    reference = samples.loc[samples["method"] == "Original", keys + ["latency_ms"]]
    reference = reference.rename(columns={"latency_ms": "original_ms"})
    for method, group in samples.groupby("method"):
        assert len(group) == len(reference), method
        assert not group.duplicated(keys).any(), method
    paired = samples.merge(reference, on=keys, how="left", validate="many_to_one")
    assert paired["original_ms"].notna().all()
    assert (paired[["original_ms", "latency_ms"]] > 0).all().all()
    paired["speedup"] = paired["original_ms"] / paired["latency_ms"]
    return paired.groupby(["method", "steps", "language", "metric"]).agg(
        excerpts=("latency_ms", "size"),
        median_ms=("latency_ms", "median"),
        q1_ms=("latency_ms", lambda values: values.quantile(0.25)),
        q3_ms=("latency_ms", lambda values: values.quantile(0.75)),
        median_speedup=("speedup", "median"),
    ).reset_index()


latency_summary = summarize_multilingual_latency(pd.DataFrame(latency_rows))
latency_summary.to_csv(OUT / "multilingual_latency_summary.csv", index=False)
display(latency_summary.round(3))

## Visual Analysis

The figures report held-out Wiki-40B perplexity across compression stages, final-stage downstream accuracy, and the one-stage calibration-language ablation. The unmodified model and held-out inputs are shared. Smoke-mode figures use small samples and serve to verify the pipeline.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

language_names = {"de": "German", "es": "Spanish", "ja": "Japanese"}
method_colors = {"Remove": "#666666", "Patch": "#0072B2", "USR": "#009E73", "Fusion": "#D55E00"}

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharex=True)
for ax, (language, name) in zip(axes, language_names.items()):
    base = wiki[(wiki.method == "Original") & (wiki.language == language)].iloc[0]
    for method in METHODS:
        rows = wiki[(wiki.method == method) & (wiki.language == language)].sort_values("steps")
        ax.plot([0] + rows["steps"].tolist(), [base["ppl"]] + rows["ppl"].tolist(),
                marker="o", label=method, color=method_colors[method])
    ax.set_title(name)
    ax.set_xlabel("Merged pairs")
    ax.set_yscale("log")
    ax.grid(True, which="both", alpha=0.25)
axes[0].set_ylabel("Wiki-40B test perplexity (log scale)")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4)
fig.suptitle(f"Perplexity across compression stages — {MODE} mode")
fig.tight_layout(rect=(0, 0.1, 1, 0.94))
fig.savefig(OUT / "perplexity_by_stage.png", dpi=180, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(13, 4.8), sharey=True)
order = ["Original", "Remove", "Patch", "USR", "Fusion"]
for ax, task in zip(axes, ("MMMLU", "INCLUDE", "MGSM")):
    subset = accuracy[accuracy.task == task]
    values = subset.pivot(index="method", columns="language", values="accuracy_pct").reindex(
        index=order, columns=list(language_names))
    counts = subset.groupby("language")["examples"].first()
    image = ax.imshow(values.to_numpy(dtype=float), vmin=0, vmax=100, cmap="Blues", aspect="auto")
    for row in range(len(order)):
        for col in range(len(language_names)):
            value = values.iloc[row, col]
            if pd.notna(value):
                ax.text(col, row, f"{value:.1f}", ha="center", va="center",
                        color="white" if value >= 60 else "black")
    ax.set_title(task)
    ax.set_xticks(range(3), [f"{name}\n(n={int(counts.get(lang, 0))})"
                             for lang, name in language_names.items()], rotation=20, ha="right")
    ax.set_yticks(range(len(order)), order)
fig.colorbar(image, ax=axes, label="Accuracy (%)", shrink=0.75)
fig.suptitle(f"Five-shot accuracy at final merge stage — {MODE} mode")
fig.savefig(OUT / "benchmark_accuracy.png", dpi=180, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(14, 4.4), sharey=True)
positions = np.arange(3)
width = 0.34
conditions = [("German only", "#E69F00"), ("Balanced multilingual", "#0072B2")]
for ax, (language, name) in zip(axes, language_names.items()):
    for index, (condition, color) in enumerate(conditions):
        values = calibration_wiki[
            (calibration_wiki.language == language) & (calibration_wiki.calibration == condition)
        ].set_index("method").reindex(["Patch", "USR", "Fusion"])
        ax.bar(positions + (index - 0.5) * width, values["ppl_change_pct"], width,
               label=condition, color=color)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(name)
    ax.set_xticks(positions, ["Patch", "USR", "Fusion"])
    ax.set_xlabel("Merge method")
    ax.grid(axis="y", alpha=0.25)
axes[0].set_ylabel("Test perplexity change vs original (%)")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2)
fig.suptitle(f"Calibration language at one merge and equal token budget — {MODE} mode")
fig.tight_layout(rect=(0, 0.1, 1, 0.94))
fig.savefig(OUT / "calibration_comparison.png", dpi=180, bbox_inches="tight")
plt.show()

## Result Files

- `revisions.json` records the model and dataset commit IDs.
- `validation_trials.csv` records candidate scores, ablations, and selections.
- `merge_plans.json` records each method's ordered merge schedule.
- `wiki_test.csv` reports per-language test perplexity and change from the unmodified checkpoint at each stage.
- `efficiency.csv` reports parameter counts, attention-block counts, and inference latency.
- `prompt_manifest.json` and `predictions.csv` record the evaluation items, prompt hashes, and individual predictions.
- `benchmark_accuracy.csv` summarizes final-stage benchmark accuracy, unparseable outputs, and differences from the reference checkpoint.

- `multilingual_latency_samples.csv` and `multilingual_latency_summary.csv` report paired per-excerpt timings and per-language latency summaries.

- benchmark_protocol.json and benchmark_sample_counts.csv record the evaluation settings and sample sizes.
- calibration_wiki.csv, calibration_predictions.csv, and calibration_accuracy.csv record the calibration-language ablation.
- perplexity_by_stage.png, benchmark_accuracy.png, and calibration_comparison.png contain labelled figures.

## References

[Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B), [Qwen3 implementation](https://github.com/huggingface/transformers/blob/v4.53.3/src/transformers/models/qwen3/modeling_qwen3.py), [Wiki-40B](https://huggingface.co/datasets/google/wiki40b), [MMMLU](https://huggingface.co/datasets/openai/MMMLU), [INCLUDE](https://huggingface.co/datasets/CohereLabs/include-base-44), [MGSM](https://huggingface.co/datasets/juletxara/mgsm).